# CEG 外部 baseline 产物 Colab Notebook

该 Notebook 只负责运行外部 baseline command plan 并保存 observations / manifest。它不实现 CEG 主方法, 也不调用 CEG-WM。


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/RICHAAARC/CEG.git"
REPO_BRANCH = ""
REPO_ROOT = Path("/content/CEG")
DRIVE_ROOT = Path("/content/drive/MyDrive/CEG")
WORKSPACE = DRIVE_ROOT / "pilot_runs" / "real_pilot_input_workspace_20260617_034500"

BASELINE_PLAN = WORKSPACE / "external_baselines" / "baseline_plan.json"
BASELINE_OUTPUT_ROOT = WORKSPACE / "external_baselines"
RUN_EXTERNAL_BASELINES = True
REQUIRE_BASELINE_PASS = True
BASELINE_FORMAL_RESULT_CLAIM = False
BASELINE_EVIDENCE_PATHS = []


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'非 Colab 环境或 Drive 已挂载: {exc}')

if not REPO_ROOT.exists():
    cmd = ['git', 'clone']
    if REPO_BRANCH:
        cmd += ['--branch', REPO_BRANCH]
    cmd += [REPO_URL, str(REPO_ROOT)]
    print('运行:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', '--all', '--prune'], check=True)
    if REPO_BRANCH:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_ROOT)], check=True)


In [ ]:
required_paths = [BASELINE_PLAN]
missing = [path for path in required_paths if not path.is_file()]
for path in required_paths:
    print(('OK' if path.is_file() else 'MISSING'), path)
if missing:
    raise FileNotFoundError('缺少外部 baseline plan: ' + ', '.join(str(path) for path in missing))
BASELINE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
cmd = [
    sys.executable,
    'scripts/run_baseline_plan.py',
    '--plan', str(BASELINE_PLAN),
    '--out', str(BASELINE_OUTPUT_ROOT),
]
if REQUIRE_BASELINE_PASS:
    cmd.append('--require-pass')
if BASELINE_FORMAL_RESULT_CLAIM:
    cmd.append('--formal-result-claim')
for evidence_path in BASELINE_EVIDENCE_PATHS:
    cmd += ['--evidence-path', str(evidence_path)]
print('运行:', ' '.join(cmd))
if RUN_EXTERNAL_BASELINES:
    subprocess.run(cmd, check=True)
else:
    print('RUN_EXTERNAL_BASELINES = False, 仅打印命令。')


In [ ]:
paths = {
    'baseline_observations': BASELINE_OUTPUT_ROOT / 'baseline_observations.json',
    'baseline_execution_manifest': BASELINE_OUTPUT_ROOT / 'baseline_execution_manifest.json',
}
for name, path in paths.items():
    print(name, path, 'exists=', path.exists())
